In [1]:
from typing import Any, Final, TypeAlias

import numpy as np

from athenspop.core.clustering.hierarchical import hierarchical_clustering

StateSequences: TypeAlias = np.ndarray[tuple[Any, Any], np.dtype[np.str_]]

state_sequences = np.load("state_sequences.npy")
distance_matrix = np.load("distance_matrix.npy")

# Verify that the distance matrix is symmetric and positive definite.
tri_diff = distance_matrix - distance_matrix.T
print(tri_diff.min())
print(tri_diff.max())

0.0
0.0


In [2]:
distance_matrix /= distance_matrix.max()
distance_matrix

array([[0.        , 0.38831583, 0.51805261, ..., 0.24993856, 0.64854658,
        0.61569515],
       [0.38831583, 0.        , 0.24096066, ..., 0.44954467, 0.48814808,
        0.33086284],
       [0.51805261, 0.24096066, 0.        , ..., 0.60437442, 0.28983045,
        0.32715969],
       ...,
       [0.24993856, 0.44954467, 0.60437442, ..., 0.        , 0.74747431,
        0.57738125],
       [0.64854658, 0.48814808, 0.28983045, ..., 0.74747431, 0.        ,
        0.48621474],
       [0.61569515, 0.33086284, 0.32715969, ..., 0.57738125, 0.48621474,
        0.        ]], shape=(512, 512))

In [3]:
squareform_matrix, linkage_matrix = hierarchical_clustering(
    distance_matrix,
    # SciPy uses a very strict threshold to validate the input matrix.
    # However, AthensPop already guarantees that the matrix is symmetric and positive definite, so this check can be disabled.
    validate_dmat_struct=False,
    optimize_leaf_order=True,
)

In [4]:
# _ = plot_clustered_distance_matrix(distance_matrix, linkage_matrix)

In [5]:
import math
from datetime import datetime, timedelta

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import scienceplots  # noqa  # noqa
import seaborn as sns

from athenspop.models.trips import TripMode, TripPurpose

FONTSIZE = 8

PHI = (1 + math.sqrt(5)) / 2

FIG_WIDTH_IN = 12.2 / 2.54
FIGSIZE = (19.3 / 2.54, 12.2 / 2.54)

mpl.rcdefaults()

plt.style.use(["science", "grid", "scatter", "ieee"])
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": FONTSIZE,
        "font.serif": [],
        "font.sans-serif": [],
        "font.cursive": [],
        "font.fantasy": [],
        "font.monospace": [],
        "text.usetex": True,
        "text.latex.preamble": "",
        # TODO: Is this necessary?
        "mathtext.fontset": "cm",
        "axes.titlesize": FONTSIZE,
        "axes.labelsize": FONTSIZE,
        "xtick.labelsize": FONTSIZE,
        "ytick.labelsize": FONTSIZE,
        "legend.fontsize": FONTSIZE,
        "legend.title_fontsize": FONTSIZE,
        "figure.titlesize": FONTSIZE,
        "figure.labelsize": FONTSIZE,
        "figure.figsize": FIGSIZE,
        "figure.dpi": 1200,
        "figure.constrained_layout.use": True,
        "savefig.format": "pgf",
        "savefig.bbox": None,
        # TODO: Is this necessary?
        "savefig.pad_inches": 0,
        "savefig.transparent": True,
        "pgf.rcfonts": False,
        "pgf.preamble": "",
        "pgf.texsystem": "pdflatex",
        # "figure.constrained_layout.h_pad": 0,
        # "figure.constrained_layout.hspace": 0,
        # "figure.constrained_layout.w_pad": 0,
        # "figure.constrained_layout.wspace": 0,
        # "font.stretch": 9.5,
        # "hatch.linewidth": 0.5,
        # "legend.borderaxespad": 0,
        # "legend.borderpad": 0,
        # "legend.fancybox": False,
        # "legend.handletextpad": 0,
        # "savefig.pad_inches": 0,
        # "xtick.major.pad": 4,
        # "xtick.minor.pad": 4,
        # "ytick.major.pad": 4,
        # "ytick.minor.pad": 4,
    }
)

sns.set_palette("colorblind")

PURP_TO_MODE_AXIS_HEIGHT_RATIO: Final[float] = 1

PURP_PALETTE: Final[dict[TripPurpose, str]] = dict(
    zip(TripPurpose, sns.color_palette("Accent"))
)
MODE_PALETTE: Final[dict[TripPurpose, str]] = dict(
    zip(TripMode, sns.color_palette("Accent"))
)

# PURP_PALETTE: Final[dict[TripPurpose, str]] = dict(
#     zip(
#         TripPurpose,
#         [
#             "#BF5553",
#             "#C76E36",
#             "#988126",
#             "#6B8E42",
#             "#2F9470",
#             "#348D9B",
#             "#5B7AC2",
#             "#8969BD",
#             "#B75796",
#             "#87695B",
#         ],
#     )
# )
# MODE_PALETTE: Final[dict[TripMode, str]] = dict(
#     zip(
#         TripMode,
#         [
#             "#CC5E49",
#             "#AF7A24",
#             "#81892E",
#             "#4B9356",
#             "#009187",
#             "#4683B0",
#             "#7071C8",
#             "#A05FB8",
#             "#C4507F",
#             "#6D7382",
#         ],
#     )
# )

TIMESTEP_FREQ_MINUTES: Final[int] = 120


def plot_state_distribution(
    state_sequences: StateSequences,
    # TODO: Replace this tuple with a typed dictionary or dataclass.
    axes: tuple[plt.Axes, plt.Axes] | None = None,
    start_hour: int = 4,
    minimal: bool = False,
) -> tuple[plt.Figure, tuple[plt.Axes, plt.Axes]]:
    sequences = pd.DataFrame(state_sequences)

    state_counts = _compute_state_counts(sequences)
    state_probs = _compute_state_probs(state_counts)
    state_probs = state_probs.rename(
        columns={column: column.replace("trip_", "") for column in state_probs.columns}
    )

    fig, (mode_axis, purp_axis) = _setup_axes(axes, minimal)

    _plot_state_distribution(
        state_counts=state_counts,
        state_probs=state_probs,
        fig=fig,
        mode_axis=mode_axis,
        purp_axis=purp_axis,
        start_hour=start_hour,
        minimal=minimal,
    )

    return fig, (mode_axis, purp_axis)


def _setup_axes(
    axes: tuple[plt.Axes, plt.Axes] | None, minimal: bool
) -> tuple[plt.Figure, tuple[plt.Axes, plt.Axes]]:
    if axes is None:
        fig, (mode_axis, purp_axis) = plt.subplots(
            nrows=2,
            ncols=1,
            sharex=True,
            # TODO: This should be a global setting or the user should handle it.
            # figsize=MINIMAL_FIG_SIZE if minimal else DEFAULT_FIG_SIZE,
            height_ratios=[PURP_TO_MODE_AXIS_HEIGHT_RATIO, 1],
            # TODO: This should be a global setting or the user should handle it.
            constrained_layout=True,
        )
    else:
        mode_axis, purp_axis = axes
        mode_axis.sharex(purp_axis)
        fig = mode_axis.figure

    return fig, (mode_axis, purp_axis)


def _compute_state_counts(state_sequences: pd.DataFrame) -> pd.DataFrame:
    return state_sequences.apply(pd.Series.value_counts).fillna(0).T


def _compute_state_probs(state_counts: pd.DataFrame) -> pd.DataFrame:
    return state_counts.div(state_counts.sum(axis=1), axis=0).fillna(0)


def _plot_state_distribution(
    # TODO: Take in len(state_counts).
    state_counts: pd.DataFrame,
    state_probs: pd.DataFrame,
    fig: plt.Figure,
    mode_axis: plt.Axes,
    purp_axis: plt.Axes,
    start_hour: int,
    minimal: bool,
) -> None:
    mode_columns = [column for column in state_probs.columns if column in set(TripMode)]
    purp_columns = [
        column for column in state_probs.columns if column in set(TripPurpose)
    ]

    mode_data = _prepare_data(state_probs[mode_columns])
    purp_data = _prepare_data(state_probs[purp_columns])

    _plot_stacked_barchart(
        mode_data,
        num_bins=len(state_counts),
        palette=MODE_PALETTE,
        legend=not minimal,
        ax=mode_axis,
        # hatch= 'o'
    )
    _plot_stacked_barchart(
        purp_data,
        num_bins=len(state_counts),
        palette=PURP_PALETTE,
        legend=not minimal,
        ax=purp_axis,
    )

    _format_axes(
        fig,
        mode_axis=mode_axis,
        purp_axis=purp_axis,
        num_timesteps=len(state_counts),
        start_hour=start_hour,
        minimal=minimal,
    )


def _prepare_data(state_probs: pd.DataFrame) -> pd.DataFrame:
    return (
        state_probs.reset_index()
        .rename(columns={"index": "time"})
        .melt(id_vars=["time"], var_name="state", value_name="freq")
    )


def _plot_stacked_barchart(
    data: pd.DataFrame,
    num_bins: int,
    palette: dict[str, str],
    legend: bool,
    ax: plt.Axes,
    # hatch=None
) -> None:
    sns.histplot(
        data,
        x="time",
        hue="state",
        weights="freq",
        bins=num_bins,
        discrete=True,
        multiple="stack",
        palette=palette,
        legend=legend,
        ax=ax,
        alpha=0.8,
        edgecolor="none",
        # hatch=hatch
    )

    ax.grid(visible=False)


def _format_axes(
    fig: plt.Figure,
    mode_axis: plt.Axes,
    purp_axis: plt.Axes,
    num_timesteps: int,
    start_hour: int,
    minimal: bool,
) -> None:
    purp_axis.set_xlabel("Time")
    purp_axis.set_ylabel("Empirical Probability")

    mode_axis.set_ylabel(None)

    if minimal:
        for ax in [mode_axis, purp_axis]:
            ax.xaxis.set_visible(False)
            ax.yaxis.set_visible(False)

        mode_axis.set_xlim(0, num_timesteps - 1)

        return

    _format_x_axis(purp_axis, start_hour=start_hour, num_timesteps=num_timesteps)
    _format_y_axis(purp_axis)

    _format_legend(mode_axis, title="Mode")
    _format_legend(purp_axis, title="Purpose")

    fig.align_ylabels([mode_axis, purp_axis])


def _format_x_axis(ax: plt.Axes, start_hour: int, num_timesteps: int) -> None:
    ax.set_xlim(0, num_timesteps - 1)

    timestep = (24 * 60) / num_timesteps
    ticks = np.arange(0, num_timesteps, TIMESTEP_FREQ_MINUTES / timestep)

    ax.set_xticks(ticks=ticks, labels=ticks)
    ax.xaxis.set_major_formatter(
        _build_time_formatter(start_hour=start_hour, num_timesteps=num_timesteps)
    )


def _format_y_axis(ax: plt.Axes) -> None:
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))


def _build_time_formatter(start_hour: int, num_timesteps: int) -> mticker.FuncFormatter:
    start_time = datetime.strptime(f"{start_hour:02d}:00", "%H:%M")
    timestep = (24 * 60) / num_timesteps

    def formatter(x: float, pos: int | None) -> str:
        if x < 0 or x >= num_timesteps:
            return ""
        time = start_time + timedelta(minutes=x * timestep)
        return time.strftime("%H:%M")

    return mticker.FuncFormatter(formatter)


def _format_legend(ax: plt.Axes, title: str) -> None:
    if ax.get_legend():
        labels = [label.get_text().title() for label in ax.legend_.texts]
        sns.move_legend(
            ax, loc="upper left", labels=labels, bbox_to_anchor=(1, 1), title=title
        )

In [8]:
import heapq
import itertools
from collections import deque
from dataclasses import dataclass

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import ClusterNode, to_tree

from athenspop.core.clustering.hierarchical import LinkageMatrix

PURP_AXIS_WIDTH: Final[float] = 1 * 0.8
PURP_AXIS_HEIGHT: Final[float] = 1 / PHI * 0.8

NODE_VERTICAL_SPACING: Final[float] = 1.5

PURP_TO_MODE_AXIS_VERTICAL_SPACING: Final[float] = 0.05


@dataclass
class Node:
    """Represents a state distribution tree node.

    This class is a soft wrapper around `scipy.cluster.hierarchy.ClusterNode` which adds the necessary layout information for plotting the tree.
    """

    linkage_node: ClusterNode
    """The corresponding node in the tree representation of the respective linkage matrix."""

    order: float = 0
    """
    The horizontal node position in the layout.
    Leaf nodes are assigned integer positions from left to right.
    Internal nodes are centered above their children, and are hence placed at half-integer coordinates.
    """

    depth: int = 0
    """
    The vertical node position in the layout.
    In contrast to standard binary trees, node depth represents its creation order, i.e., the corresponding 'split' in the respective dendrogram.
    The root has a depth of 0 and each subsequent split creates nodes at increasing depths.
    """

    is_leaf: bool = False
    """
    A flag indicating whether the node should be plotted as a leaf.
    Leaf nodes are either singleton clusters in the corresponding dendrogram or nodes whose linkage distance is less than the respective threshold for the specified number of clusters.
    """

    left: "Node | None" = None
    """The left child node."""

    right: "Node | None" = None
    """The right child node."""


def plot_state_distribution_tree(
    linkage_matrix: LinkageMatrix, state_sequences: StateSequences, num_clusters: int
) -> plt.Figure:
    linkage_dist = _compute_linkage_dist(linkage_matrix, num_clusters)

    # This data structure acts as a shared cache amongst the tree branches.
    leaf_counter = itertools.count()
    root = _build_tree(
        linkage_root=to_tree(linkage_matrix),
        linkage_dist=linkage_dist,
        leaf_counter=leaf_counter,
    )
    _assign_node_depths(root)

    fig = _setup_figure(root, num_leaves=next(leaf_counter))

    _plot_state_distribution_tree(
        state_sequences, fig, root, linkage_matrix[:, 2].max()
    )

    _plot_legend(fig.gca())

    return fig


def _compute_linkage_dist(linkage_matrix: LinkageMatrix, num_clusters: int) -> float:
    num_samples = linkage_matrix.shape[0] + 1
    if num_clusters >= num_samples:
        # "Cut" the bottom of the dendrogram.
        return 0
    if num_clusters <= 1:
        # "Cut" the top of the dendrogram.
        return linkage_matrix[-1, 2] + 1

    # The linkage matrix is sorted by merge distance.
    # Therefore, given $n$ samples and $k$ clusters, we compute the linkage distance by "reverting" the last $(n - k)$ merges, keeping the merge at the $(n - k)$-th index.
    # The resulting linkage distance which corresponds to the specified number of clusters.
    merge_index = num_samples - num_clusters

    # Add a small epsilon to ensure strict inequality works correctly.
    # Alternatively, use the midpoint to the next merge.
    return linkage_matrix[merge_index, 2]


# TODO: Implement this function without recursion.
def _build_tree(
    linkage_root: ClusterNode, linkage_dist: float, leaf_counter: itertools.count
) -> Node:
    is_true_leaf = linkage_root.is_leaf()
    is_plot_leaf = linkage_root.dist < linkage_dist

    root = Node(linkage_node=linkage_root, is_leaf=is_true_leaf or is_plot_leaf)

    if root.is_leaf:
        root.order = next(leaf_counter)
        return root

    root.left = _build_tree(linkage_root.left, linkage_dist, leaf_counter)
    root.right = _build_tree(linkage_root.right, linkage_dist, leaf_counter)

    root.order = (root.left.order + root.right.order) / 2

    return root


def _assign_node_depths(root: Node) -> None:
    root.depth = 0

    if root.is_leaf:
        return

    curr_depth = 0
    # The tie breaker ensures deterministic ordering when two node pairs have the same linkage distance.
    tie_breaker = 0
    pq = [
        (
            # Primary Key
            -root.linkage_node.dist,
            # Backup Key
            tie_breaker,
            # Value
            root,
        )
    ]
    while pq:
        _, _, node = heapq.heappop(pq)

        curr_depth += 1
        for child in [node.left, node.right]:
            child.depth = curr_depth

            if not child.is_leaf:
                tie_breaker += 1
                heapq.heappush(pq, (-child.linkage_node.dist, tie_breaker, child))


# TODO: Implement this function without recursion.
def _compute_tree_depth(root: Node | None) -> int:
    if root is None:
        return 0

    left_depth = _compute_tree_depth(root.left)
    right_depth = _compute_tree_depth(root.right)

    return max(root.depth, left_depth, right_depth)


# TODO: Rename this function to _setup_axises.
def _setup_figure(layout_root: Node, num_leaves: int) -> plt.Figure:
    fig = plt.figure(
        # figsize=DEFAULT_FIG_SIZE
    )
    axis = fig.add_axes((0, 0, 1, 1))
    axis.axis("off")

    depth = _compute_tree_depth(layout_root)

    half_node_width = PURP_AXIS_WIDTH / 2
    half_node_height = PURP_AXIS_HEIGHT / 2

    # TODO: Document these calculations.
    x_min = -half_node_width
    x_max = half_node_width + num_leaves - 1

    y_min = -half_node_height - depth * NODE_VERTICAL_SPACING
    y_max = (
        half_node_height
        + PURP_AXIS_HEIGHT * PURP_TO_MODE_AXIS_HEIGHT_RATIO
        + PURP_TO_MODE_AXIS_VERTICAL_SPACING
    )

    axis.set_xlim(x_min, x_max)
    axis.set_ylim(y_min, y_max)

    return fig


def _plot_state_distribution_tree(
    state_sequences: StateSequences,
    fig: plt.Figure,
    root: Node,
    max_linkage_dist: float,
) -> None:
    axis = fig.gca()

    queue = deque([root])
    while queue:
        node = queue.popleft()

        _plot_node(node, state_sequences, axis, max_linkage_dist)

        if not node.is_leaf:
            _plot_edges(node, axis)

            queue.append(node.left)
            queue.append(node.right)


def _plot_node(
    node: Node, sequences: StateSequences, ax: plt.Axes, max_linkage_dist: float
) -> None:
    half_purp_axis_width = PURP_AXIS_WIDTH / 2
    half_purp_axis_height = PURP_AXIS_HEIGHT / 2

    x_min = node.order - half_purp_axis_width
    y_min = -node.depth * NODE_VERTICAL_SPACING - half_purp_axis_height

    mode_axis_height = PURP_AXIS_HEIGHT * PURP_TO_MODE_AXIS_HEIGHT_RATIO
    purp_axis_height = PURP_AXIS_HEIGHT

    mode_axis = ax.inset_axes(
        (
            x_min,
            y_min + purp_axis_height + PURP_TO_MODE_AXIS_VERTICAL_SPACING,
            PURP_AXIS_WIDTH,
            mode_axis_height,
        ),
        transform=ax.transData,
        zorder=10,
    )
    purp_axis = ax.inset_axes(
        (x_min, y_min, PURP_AXIS_WIDTH, purp_axis_height),
        transform=ax.transData,
        zorder=10,
    )

    cluster_members = node.linkage_node.pre_order()
    cluster_sequences = sequences[cluster_members]

    plot_state_distribution(
        state_sequences=cluster_sequences, axes=(mode_axis, purp_axis), minimal=True
    )

    title = f"ID: {node.order + 1}\n" if node.is_leaf else ""
    mode_axis.set_title(
        title
        + r"$\tilde{H}$"
        + f": {node.linkage_node.dist / max_linkage_dist:.2f}"
        + f"\nSize: {len(cluster_sequences)}",
        loc="left",
    )


def _plot_edges(root: Node, ax: plt.Axes) -> None:
    half_purp_axis_height = PURP_AXIS_HEIGHT / 2

    y_max = -root.depth * NODE_VERTICAL_SPACING - half_purp_axis_height
    for child in [root.left, root.right]:
        if child is None:
            continue

        y_min = (
            -child.depth * NODE_VERTICAL_SPACING
            + half_purp_axis_height
            + PURP_TO_MODE_AXIS_VERTICAL_SPACING
            + PURP_AXIS_HEIGHT * PURP_TO_MODE_AXIS_HEIGHT_RATIO
        )
        y_mid = (y_min + y_max) / 2

        ax.plot(
            # Plot a line from the root node to its children.
            [root.order, root.order, child.order, child.order],
            # Split the line at the midpoint between the root and each child.
            [y_max, y_mid, y_mid, y_min],
            alpha=0.4,
            color="#000000",
            linestyle="-",
            # Place the line below the nodes.
            zorder=1,
        )


# TODO: Rename this function to _format_legend.
def _plot_legend(axis: plt.Axes) -> None:
    purp_patches = [
        mpatches.Patch(color=color, label=label.title())
        for label, color in PURP_PALETTE.items()
    ]
    mode_patches = [
        mpatches.Patch(color=color, label=label.title())
        for label, color in MODE_PALETTE.items()
    ]

    purp_legend = axis.legend(
        handles=purp_patches,
        loc="lower left",
        bbox_to_anchor=(0, 0),
        title="Activity Purpose",
    )
    axis.add_artist(purp_legend)

    # Compute the legend bounds.
    axis.figure.canvas.draw()

    renderer = axis.figure.canvas.get_renderer()
    purp_legend_bbox = purp_legend.get_window_extent(renderer=renderer).transformed(
        axis.transAxes.inverted()
    )

    axis.legend(
        handles=mode_patches,
        loc="lower left",
        bbox_to_anchor=(
            0
            + purp_legend_bbox.width
            # TODO: Tune the horizontal legend spacing.
            + 0.01,
            0,
        ),
        title="Travel Mode",
    )

In [9]:
from typing import Final

NUM_CLUSTERS: Final[int] = 10

tree = plot_state_distribution_tree(
    linkage_matrix, state_sequences, num_clusters=NUM_CLUSTERS
)
# tree.set_dpi(1200)
# tree.set_dpi(300)

C:\Users\Dimit\AppData\Local\Temp\ipykernel_13636\2341627026.py:300: UserWarning: There are no gridspecs with layoutgrids. Possibly did not call parent GridSpec with the "figure" keyword
  axis.figure.canvas.draw()
C:\Documents\athenspop\.venv\Lib\site-packages\IPython\core\events.py:96: UserWarning: There are no gridspecs with layoutgrids. Possibly did not call parent GridSpec with the "figure" keyword
  func(*args, **kwargs)
C:\Documents\athenspop\.venv\Lib\site-packages\IPython\core\pylabtools.py:170: UserWarning: There are no gridspecs with layoutgrids. Possibly did not call parent GridSpec with the "figure" keyword
  fig.canvas.print_figure(bytes_io, **kw)


In [12]:
tree.savefig(
    "figures/DendrogramFinal.eps", bbox_inches="tight", orientation="landscape"
)

C:\Users\Dimit\AppData\Local\Temp\ipykernel_13636\4098794297.py:1: UserWarning: There are no gridspecs with layoutgrids. Possibly did not call parent GridSpec with the "figure" keyword
  tree.savefig("figures/DendrogramFinal.eps",
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
Failed to find a Ghostscript installation.  Distillation step skipped.


In [ ]:
import scipy as sp

labels = sp.cluster.hierarchy.fcluster(linkage_matrix, t=10, criterion="maxclust")
for label in np.unique(labels):
    plot_state_distribution(state_sequences[labels == label])
    plt.savefig(f"figures/{label}.png")

In [24]:
import scipy as sp

# labels=sp.cluster.hierarchy.fcluster(linkage_matrix,t=10,criterion="maxclust")
# for label in np.unique(labels):
plot_state_distribution(state_sequences)
plt.savefig("figures/root.png")

In [ ]:
def compute_cluster_memoids(
    distance_matrix: DistanceMatrix, labels: ClusterLabels
) -> list[np.int64]:
    memoids = []
    for label in np.unique(labels):
        partition, members = _partition_distance_matrix(distance_matrix, labels, label)
        memoid = _compute_memoid(partition, members)

        memoids.append(memoid)

    return memoids


def _partition_distance_matrix(
    distance_matrix: DistanceMatrix, labels: ClusterLabels, label: int
) -> tuple[DistanceMatrix, ClusterLabels]:
    members = np.where(labels == label)[0]
    partition = distance_matrix[np.ix_(members, members)]

    return partition, members


ClusterMembers: TypeAlias = np.ndarray[tuple[Any], np.dtype[np.int64]]


def _compute_memoid(
    partitioned_distance_matrix: DistanceMatrix, members: ClusterMembers
) -> np.int64:
    distance_to_neighbors = partitioned_distance_matrix.sum(axis=1)

    local_index = np.argmin(distance_to_neighbors)
    global_index = members[local_index]

    return global_index


memoids = compute_cluster_memoids(distance_matrix, labels)
memoids

In [ ]:
# TODO: inconsistent, maxinconsts, maxdists, maxRstat, disjointSet
# TODO: validity predicates - what?

In [ ]:
# TODO: add a function that plots a single sequence
# TODO: add shankey diagrams and this sicrular shankey diagram thing
# TODO: implement silhouette & umap plot